# 1D step potential — prepared circuit and sampled results

A moving Gaussian incident on the exact discrete step $V(x)=1.5$ for $x\geq0$. Units: $m=\hbar=1$.

## Direct construction

No repository helper imports. Every preparation, phase, QFT, and measurement routine used here is defined below.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from qiskit import ClassicalRegister, QuantumCircuit, QuantumRegister, transpile
from qiskit.circuit.library import DiagonalGate, QFTGate
from qiskit_aer import AerSimulator
from qiskit.visualization import plot_histogram

def periodic_grid(num_qubits, length):
    """N=2**n endpoint-excluded samples on [-L/2, L/2)."""
    return np.linspace(-length / 2, length / 2, 2**num_qubits, endpoint=False)

def gaussian_packet(x, x0, sigma, k0):
    """Normalized discrete position amplitudes for the initialization gate."""
    psi = np.exp(-(x - x0)**2 / (4 * sigma**2)) * np.exp(1j * k0 * x)
    return psi / np.linalg.norm(psi)

def fft_wavenumbers(x):
    return 2 * np.pi * np.fft.fftfreq(len(x), d=x[1] - x[0])

def qft_split_step_circuit(psi, x, potential, dt, steps=1):
    """Prepare |psi>, then apply V(dt/2), QFT, T(dt), QFT†, V(dt/2)."""
    n = int(np.log2(len(psi)))
    position = QuantumRegister(n, 'x')
    circuit = QuantumCircuit(position, name='split_step')
    circuit.initialize(psi, position)
    v_phase = np.exp(-0.5j * potential * dt)
    k_phase = np.exp(-0.5j * fft_wavenumbers(x)**2 * dt)
    for _ in range(steps):
        circuit.append(DiagonalGate(v_phase), position)
        circuit.append(QFTGate(n), position)
        circuit.append(DiagonalGate(k_phase), position)
        circuit.append(QFTGate(n).inverse(), position)
        circuit.append(DiagonalGate(v_phase), position)
    return circuit, position

def measure_position(circuit, position, shots=8192):
    """Run local shot sampling; include all integer outcomes."""
    measured = circuit.copy()
    classical = ClassicalRegister(len(position), 'position')
    measured.add_register(classical)
    measured.measure(position, classical)
    backend = AerSimulator()
    compiled = transpile(measured, backend, optimization_level=0)
    raw = backend.run(compiled, shots=shots).result().get_counts()
    counts = {i: raw.get(format(i, f'0{len(position)}b'), 0) for i in range(2**len(position))}
    return measured, counts


In [ ]:
# Parameters: small enough for a visible circuit and local-shot experiment.
n, length, dt, steps, shots = 5, 8.0, 0.04, 25, 8192
x = periodic_grid(n, length)
potential = np.where(x >= 0.0, 1.5, 0.0)
psi0 = gaussian_packet(x, x0=-2.0, sigma=0.55, k0=2.0)
evolution_circuit, position = qft_split_step_circuit(psi0, x, potential, dt, steps)
measured_circuit, counts = measure_position(evolution_circuit, position, shots)
probabilities = np.array([counts[i] / shots for i in range(2**n)])
print(f'{shots:,} local Aer shots; {len(evolution_circuit.data)} coherent operations before measurement.')

## Read the circuit before reading the result

This is the actual circuit sent to the local quantum simulator. Initialize loads the sampled wavefunction. Each repeated block applies a position-space potential phase, QFT, momentum-space kinetic phase, inverse QFT, then the final potential half-phase. Measurement occurs only after all coherent steps.

The QFT box stays compact; expanding it adds the usual controlled-phase network without improving the physics story. Run evolution_circuit.decompose(reps=1).draw('text') to inspect it.

In [ ]:
# Clean map of one actual Strang step. `one_step` retains the exact gate parameters.
from qiskit.circuit import Gate
one_step, _ = qft_split_step_circuit(psi0, x, potential, dt)
preparation_map = QuantumCircuit(n)
for label in ('Initialize ψ₀', 'V(Δt/2)', 'QFT', 'T(Δt)', 'QFT†', 'V(Δt/2)'):
    preparation_map.append(Gate(label, n, []), range(n))
print(preparation_map.draw('text', fold=120))
print('Exact one-step operations:', one_step.count_ops())

## Results: sampled circuit measurements

In [ ]:
# Bars: frequencies from sampled circuit measurement, not a NumPy |psi|² plot.
fig, ax = plt.subplots(figsize=(11, 4))
ax.bar(x, probabilities, width=(x[1] - x[0]) * .9, color='#4C78A8', label='Aer measurement frequency')
if potential.max() > 0:
    ax.plot(x, potential / potential.max() * max(probabilities.max() * 1.15, .02),
            'r--', lw=2, label='step: V(x)=1.5 for x≥0 (scaled)')
ax.set(title='Measured position distribution after t = {dt * steps:g}',
       xlabel='position sample x[j]', ylabel='counts / shots')
ax.legend()
plt.show()

# Same measured data in Qiskit's computational-basis histogram.
plot_histogram({format(i, f'0{n}b'): count for i, count in counts.items()},
               title='Shot counts by computational-basis position state', figsize=(12, 4))

## What is quantum here; what is classical here

gaussian_packet classically calculates the amplitudes required by the initialize gate. Everything after initialization is assembled from Qiskit gates. The result plots come from 8,192 circuit measurements, so they contain shot noise.

This is ideal local Aer simulation, not hardware data: no calibration noise, routing, or imperfect preparation. The grid is periodic; stop before the packet wraps across an edge.